# **Big Data Analytics [CN7031] CRWK 2025-26**


# **Group ID: 280**
1.   Student 1: Ranjeet Kumar Mahato 3035175
2.   Student 2: Janak Mahato 3033365
3.   Student 3: Irfan Khan 3033571
4.   Student 4: Muhammad Noman Yousaf 3031756

---

#**Student 1: Ranjeet Kumar Mahato 3035175**

# **Initiate and Configure Spark**

---

In [ ]:
!pip install --upgrade pip
!pip install datasets==2.19.0 pyspark==3.5.1 pandas==2.2.2


In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, regexp_extract, split, size, length, when, lit, floor, rand, expr, broadcast, year
from pyspark.sql.functions import avg, count as spark_count
from pyspark.sql.types import IntegerType

spark = SparkSession.builder \
    .appName("CN7031_Amazon_Apps_Android") \
    .config("spark.sql.shuffle.partitions", "200") \
    .config("spark.driver.memory", "4g") \
    .config("spark.executor.memory", "2g") \
    .getOrCreate()

print("Spark version:", spark.version)


Spark version: 4.0.1


#**Stream the Hugging Face Semi-Sructured Data from Amazon-Review-2023 and create a local JSONL ≥200MB**

In [2]:
from google.colab import drive
drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
# Stream subset of the large dataset to local JSONL until file size >= 210MB.
from datasets import load_dataset
import os, json


CONFIG_NAME  = "raw_review_All_Beauty"
OUTPUT_PATH = "Amazon_raw_review_All_Beauty.jsonl"
TARGET_SIZE_BYTES = 210 * 1024 * 1024  # 210MB

if os.path.exists(OUTPUT_PATH) and os.path.getsize(OUTPUT_PATH) >= TARGET_SIZE_BYTES:
    print("File already exists:", OUTPUT_PATH, "size MB:", os.path.getsize(OUTPUT_PATH)/1024/1024)
else:
    print("Streaming dataset split:", CONFIG_NAME )
    dataset_stream = load_dataset("McAuley-Lab/Amazon-Reviews-2023", CONFIG_NAME , streaming=True, trust_remote_code=True)
    written = 0
    with open(OUTPUT_PATH, "w", encoding="utf-8") as f:
      for record in dataset_stream:
        f.write(json.dumps(record, ensure_ascii=False) + "\n")
        written_records += 1

        # Check file size every 1000 records
        if written_records % 1000 == 0:
            current_size = os.path.getsize(OUTPUT_PATH)
            current_size_mb = current_size / (1024 * 1024)
            print(f"{written_records} records written — {current_size_mb:.2f} MB")

        # Stop once we exceed 200 MB
        if os.path.getsize(OUTPUT_PATH) >= TARGET_SIZE_BYTES:
            print("Target size reached. Stopping stream.")
            break
    print("Finished writing amazon_reviews_subset.json records:", written, "size MB:", os.path.getsize(OUTPUT_PATH)/1024/1024)


Streaming dataset split: raw_review_All_Beauty
Finished writing amazon_reviews_subset.jsonl records: 1 size MB: 6.67572021484375e-06


#**TASK 1 - Hierarchical Data Parsing & Flattening (ID 1)**

**Objective:** Process a large semistructured dataset (e.g., JSON or XML). Design and implement a
PySpark pipeline (using either RDD operations or advanced
DataFrame functions like explode, struct, or custom UDFs) to
flatten the data into an analytic structure.

##Load Semi-Structured Dataset into PySpark (infer schema)

In [ ]:
# Run in Python cell
save_path = "/content/drive/MyDrive/Amazon_raw_review_All_Beauty.jsonl"
df = spark.read.json(save_path)
print("Rows in saved file:", df.count())
df.printSchema()
df.show(5, truncate=200)


Rows in saved file: 701528
root
 |-- asin: string (nullable = true)
 |-- helpful_vote: long (nullable = true)
 |-- images: array (nullable = true)
 |    |-- element: struct (containsNull = true)
 |    |    |-- attachment_type: string (nullable = true)
 |    |    |-- large_image_url: string (nullable = true)
 |    |    |-- medium_image_url: string (nullable = true)
 |    |    |-- small_image_url: string (nullable = true)
 |-- parent_asin: string (nullable = true)
 |-- rating: double (nullable = true)
 |-- text: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- title: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- verified_purchase: boolean (nullable = true)

+----------+------------+------+-----------+------+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-------------+---------------------------

In [ ]:
print(df.columns)

['asin', 'helpful_vote', 'images', 'parent_asin', 'rating', 'text', 'timestamp', 'title', 'user_id', 'verified_purchase']


## Flatten the nested `images` array using `explode`
This converts each image struct into its own row and extracts its fields.


In [ ]:
from pyspark.sql.functions import explode, col

df_flat = df.withColumn("image", explode("images")) \
            .select(
                "asin",
                "parent_asin",
                "helpful_vote",
                "rating",
                "text",
                "timestamp",
                "title",
                "user_id",
                "verified_purchase",
                col("image.attachment_type").alias("attachment_type"),
                col("image.large_image_url").alias("large_image_url"),
                col("image.medium_image_url").alias("medium_image_url"),
                col("image.small_image_url").alias("small_image_url")
            )


In [ ]:
df_flat.printSchema()
df_flat.show(10, truncate=120)


root
 |-- asin: string (nullable = true)
 |-- parent_asin: string (nullable = true)
 |-- helpful_vote: long (nullable = true)
 |-- rating: double (nullable = true)
 |-- text: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- title: string (nullable = true)
 |-- user_id: string (nullable = true)
 |-- verified_purchase: boolean (nullable = true)
 |-- attachment_type: string (nullable = true)
 |-- large_image_url: string (nullable = true)
 |-- medium_image_url: string (nullable = true)
 |-- small_image_url: string (nullable = true)

+----------+-----------+------------+------+------------------------------------------------------------------------------------------------------------------------+-------------+----------------------------------+----------------------------+-----------------+---------------+-------------------------------------------------------------------------+------------------------------------------------------------------------+---------------------

#**TASK 2 - Advanced REGEX Feature Extractions ( ID 4)**
**Objective:** we extract meaningful linguistic features from unstructured review text.
We combine the title and review body, normalize text, and apply multiple advanced regular
expressions using `regexp_extract`.

##Combine & Normalize Text

In [ ]:
from pyspark.sql.functions import (
    col, regexp_extract, regexp_replace, trim, lower, length, concat_ws
)

# Combine title + text
df_re = df.withColumn(
    "full_text",
    trim(concat_ws(" ", col("title"), col("text")))
)

Extract Sentiment Words, Negation Phrases, Smell-Related Phrases, Review Length Categorization

In [ ]:

# 1️⃣ Sentiment Words (positive/negative)
df_re = df_re.withColumn(
    "sentiment_word",
    regexp_extract(
        lower(col("full_text")),
        r"\b(amazing|excellent|great|love|good|bad|terrible|awful|poor|disappointed|happy|unhappy)\b",
        1
    )
)

# 2️⃣ Negation Patterns (“not good”, “not bad”, “never buying again”)
df_re = df_re.withColumn(
    "negation_phrase",
    regexp_extract(
        lower(col("full_text")),
        r"\b(not\s+\w+|never\s+\w+)\b",
        1
    )
)

# 3️⃣ Extract smell-related descriptors (common in beauty reviews)
df_re = df_re.withColumn(
    "smell_phrase",
    regexp_extract(
        lower(col("full_text")),
        r"\b(smells\s+\w+|fragrance|scent|odor)\b",
        1
    )
)


# 6️⃣ Extract review length category
df_re = df_re.withColumn(
    "review_length",
    when(length(col("text")) < 50, "short")
    .when(length(col("text")) < 150, "medium")
    .otherwise("long")
)

# Show results
df_re.select(
    "asin", "title",
    "sentiment_word", "negation_phrase", "smell_phrase", "review_length"
).show(25, truncate=False)


+----------+---------------------------------------------------------------+--------------+----------------+------------+-------------+
|asin      |title                                                          |sentiment_word|negation_phrase |smell_phrase|review_length|
+----------+---------------------------------------------------------------+--------------+----------------+------------+-------------+
|B00YQ6X8EO|Such a lovely scent but not overpowering.                      |good          |not overpowering|scent       |long         |
|B081TJ8YS3|Works great but smells a little weird.                         |great         |                |smells a    |long         |
|B07PNNCSP9|Yes!                                                           |good          |                |smells good |short        |
|B09JS339BZ|Synthetic feeling                                              |              |                |            |short        |
|B08BZ63GMJ|A+                                  

#**TASK3 - Range Partitioning for Search Efficiency (RDD/DF) (ID 14):**
**Objective:** Apply complex REGEX
patterns within a PySpark pipeline (via RDD map/filter or DF UDFs)
to extract sophisticated features from an unstructured text
column.

Our dataset contains:

- rating (0.0–5.0)

- timestamp (epoch milliseconds)

- Other review text fields

Range queries are common:

- “Find products with rating between X and Y”

- “Find reviews posted in a time period”

- “Find top-rated items per score range”

Hash partitioning randomizes rows, so Spark must scan all partitions for range queries - slow.

Range partitioning sorts keys into ordered partitions - Spark touches only relevant partitions.

##Prepare the Dataset


Use df_clean for ALL downstream tasks

In [ ]:
from pyspark.sql.functions import *
from pyspark.sql.types import *

# Create clean version of dataset
df_clean = df.withColumn("text_clean", trim(col("text"))) \
             .withColumn("title_clean", trim(col("title"))) \
             .withColumn("meta_text",
                         when(length(col("text_clean")) > 0, col("text_clean"))
                         .otherwise(col("title_clean"))
             ) \
             .withColumn("helpful_vote", col("helpful_vote").cast("int")) \
             .withColumn("rating", col("rating").cast("double"))

df_clean.select("asin", "title_clean", "meta_text", "helpful_vote", "rating").show(5, truncate=120)


+----------+-----------------------------------------+------------------------------------------------------------------------------------------------------------------------+------------+------+
|      asin|                              title_clean|                                                                                                               meta_text|helpful_vote|rating|
+----------+-----------------------------------------+------------------------------------------------------------------------------------------------------------------------+------------+------+
|B00YQ6X8EO|Such a lovely scent but not overpowering.|This spray is really nice. It smells really good, goes on really fine, and does the trick. I will say it feels like y...|           0|   5.0|
|B081TJ8YS3|   Works great but smells a little weird.|This product does what I need it to do, I just wish it was odorless or had a soft coconut smell. Having my head smell...|           1|   4.0|
|B07PNNCSP9|        

We create a clean numeric column for partitioning:

In [ ]:
from pyspark.sql.functions import col

df_range = df_clean.withColumn(
    "rating_number", col("rating").cast("float")
).select("rating_number", "parent_asin")

df_range.show(5)


+-------------+-----------+
|rating_number|parent_asin|
+-------------+-----------+
|          5.0| B00YQ6X8EO|
|          4.0| B081TJ8YS3|
|          5.0| B097R46CSY|
|          1.0| B09JS339BZ|
|          5.0| B08BZ63GMJ|
+-------------+-----------+
only showing top 5 rows


##Create a Range-Partitioned Version

We use repartitionByRange() + sortWithinPartitions():

In [ ]:
df_range_part = df_range.repartitionByRange(
    10, "rating_number"
).sortWithinPartitions("rating_number")

df_range_part.cache().count()


701528

Sorted partitions - Spark can skip partitions faster - optimized range search.

##Compare performance: Unpartitioned vs Range-Partitioned

We compare performance for a range query:

In [ ]:
import time

# Unpartitioned search
t0 = time.time()
unpart_count = df_range.filter("rating_number BETWEEN 3.0 AND 5.0").count()
t1 = time.time()

print("Unpartitioned time:", t1 - t0)

# Range-partitioned search
t2 = time.time()
part_count = df_range_part.filter("rating_number BETWEEN 3.0 AND 5.0").count()
t3 = time.time()

print("Range-partitioned time:", t3 - t2)


Unpartitioned time: 3.3908398151397705
Range-partitioned time: 0.8494772911071777


##Observation
| Version                  | Expected Performance                                                  |
| ------------------------ | --------------------------------------------------------------------- |
| **Unpartitioned DF**     | **Slow**: must scan all partitions                                    |
| **Range-partitioned DF** | **Much faster**: Spark only scans partitions whose ranges overlap 3–5 |


###Why Range Partitioning Outperforms Hash Partitioning

Hash partitioning distributes rows using a hash function:

   partition = hash(key) % numPartitions

This produces even distribution but destroys ordering.
Therefore, a query such as:

WHERE rating BETWEEN 3.0 AND 5.0

must scan every partition, because rows are randomly spread.

Range partitioning distributes rows based on value ranges, so each partition holds a contiguous slice:

| Partition | rating_number Range |
| --------- | ------------------- |
| P0        | 0.0–0.5             |
| P1        | 0.5–1.0             |
| …         | …                   |
| P9        | 4.5–5.0             |

This makes Spark able to:
- Skip partitions that are out of range
- Prune tasks automatically
- Reduce shuffle and scan time

Thus, range partitioning is superior for ordered queries, especially in datasets like Amazon reviews where rating and timestamp are natural range-based keys.


#**TASK4 - Implementing a Custom Aggregator (User Defined Aggregate Function - UDAF) (ID 17)**
**Objective:** To design a custom User-Defined Aggregate Function (UDAF) using PySpark’s modern API (Pandas Grouped Aggregate UDF) and apply it to the Amazon Reviews dataset.

**Why this task is appropriate for our dataset:**  
Amazon reviews contain:
- `rating` (numerical)
- `helpful_vote` (weight)
- Many reviews per product (`asin`)

This makes the dataset perfectly suitable for computing a **Weighted Average Rating**, where reviews with more helpful votes should count more heavily.

This task demonstrates:
- Custom aggregation logic  
- Pandas UDF (GROUPED_AGG)  
- Comparison with built-in Spark SQL aggregation  
- Performance and clarity differences

##Define the Custom UDAF

In [ ]:
import pyspark.sql.functions as F
from pyspark.sql.types import DoubleType
from pyspark.sql import SparkSession
from pyspark.sql.functions import pandas_udf, PandasUDFType
import pandas as pd

@pandas_udf(DoubleType(), PandasUDFType.GROUPED_AGG)
def weighted_avg_rating(rating: pd.Series, helpful: pd.Series) -> float:
    total_weight = helpful.sum()
    if total_weight == 0:
        return None
    return (rating * helpful).sum() / total_weight



/usr/local/lib/python3.12/dist-packages/pyspark/sql/pandas/functions.py:446: UserWarning: In Python 3.6+ and Spark 3.0+, it is preferred to specify type hints for pandas UDF instead of specifying pandas UDF type which will be deprecated in the future releases. See SPARK-28264 for more details.
  warnings.warn(


### **Understanding the Custom Aggregator**

We created a custom **Weighted Average Rating**:


- Reviews with more helpful votes contribute more.
- If a product has only unhelpful reviews (sum of helpful votes = 0), we return `None`.

We implement this using a **Pandas Grouped Aggregate UDF**, the recommended way to build UDAFs in PySpark 3.x.


### **Applying the Custom Weighted Average Aggregator**

We use `groupBy("asin")` to compute the weighted average rating for every product in the dataset.


In [ ]:
df_weighted = df.groupBy("asin").agg(
    weighted_avg_rating(
        F.col("rating").cast("double"),
        F.col("helpful_vote").cast("double")
    ).alias("weighted_rating")
)

df_weighted.show(10, truncate=False)


+----------+---------------+
|asin      |weighted_rating|
+----------+---------------+
|0005946468|NULL           |
|0123034892|NULL           |
|0124784577|5.0            |
|0764490117|5.0            |
|0816091846|4.55           |
|0963416391|5.0            |
|0976651246|NULL           |
|0977217213|5.0            |
|0977949125|5.0            |
|097920500X|5.0            |
+----------+---------------+
only showing top 10 rows


##Comparison With Built-in Functions

### **Comparison with Built-in Spark Aggregation**

To illustrate the value of UDAFs, we compute the same result using only built-in Spark functions.


##Built-in Aggregation

In [ ]:
df_builtin = df.groupBy("asin").agg(
    F.try_divide(
        F.sum(F.col("rating") * F.col("helpful_vote")),
        F.sum("helpful_vote")
    ).alias("builtin_weighted_rating")
)

df_builtin.show(10, truncate=False)


+----------+-----------------------+
|asin      |builtin_weighted_rating|
+----------+-----------------------+
|B08ZSR9HLF|4.818181818181818      |
|B094VG2GKR|NULL                   |
|B07K1LGWYB|5.0                    |
|B0B8DB17VT|4.0                    |
|B085M95TX2|4.375                  |
|B07YS9W97B|4.75                   |
|B00TK0VV68|NULL                   |
|B00VUIVWA8|NULL                   |
|B07MMWXPLJ|3.432                  |
|B07HCR8QDN|NULL                   |
+----------+-----------------------+
only showing top 10 rows


##Compare Results

In [ ]:
df_compare = df_weighted.join(df_builtin, "asin")

df_compare.show(10, truncate=False)


+----------+---------------+-----------------------+
|asin      |weighted_rating|builtin_weighted_rating|
+----------+---------------+-----------------------+
|0005946468|NULL           |NULL                   |
|0123034892|NULL           |NULL                   |
|0124784577|5.0            |5.0                    |
|0764490117|5.0            |5.0                    |
|0816091846|4.55           |4.55                   |
|0963416391|5.0            |5.0                    |
|0976651246|NULL           |NULL                   |
|0977217213|5.0            |5.0                    |
|0977949125|5.0            |5.0                    |
|097920500X|5.0            |5.0                    |
+----------+---------------+-----------------------+
only showing top 10 rows


### **Interpretation of Results**

- The custom UDAF and built-in Spark SQL aggregation should produce **identical values**.
- This validates our custom UDAF implementation.
- Using Pandas UDFs allows:
  - Complex or non-standard aggregation logic
  - Python-based logic integrated into Spark
  - Faster runtime than old UDAF API

### **When is a Custom UDAF Useful?**

Built-in functions fail when:
- You need trimmed means
- Weighted medians
- Domain-specific scoring
- Complex normalization
- Trend-based aggregation
- Probability models inside aggregation

